# 03 — Known Authors and Public Account Mapping

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("03_known_authors_mapping", total_steps=5, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook maps the public/known author file to obfuscated author IDs, then summarizes public account activity, follower reach, and verification behavior.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
known = pd.read_csv(RAW / "well_known_authors_philippine_elections.csv")
tweets = pd.read_csv(PROCESSED / "tweets_cleaned_base.csv", parse_dates=["createdAt"])
print("Known authors:", known.shape)
print("Tweets:", tweets.shape)
display(known.head())


In [ ]:
progress.step("Purpose")
known["pseudo_author_key"] = known["obfuscated_userName"].astype(str).str.replace("@", "", regex=False)
tweets["pseudo_author_key"] = tweets["pseudo_author_userName"].astype(str).str.replace("@", "", regex=False)

author_activity = tweets.groupby("pseudo_author_key", as_index=False).agg(
    tweet_count=("pseudo_id", "count"),
    total_views=("viewCount", "sum"),
    total_likes=("likeCount", "sum"),
    total_retweets=("retweetCount", "sum"),
    total_replies=("replyCount", "sum"),
    total_quotes=("quoteCount", "sum"),
    avg_views=("viewCount", "mean"),
    verified_in_dataset=("author_isBlueVerified", "max"),
)

known_summary = known.merge(author_activity, on="pseudo_author_key", how="left")
known_summary["tweet_count"] = known_summary["tweet_count"].fillna(0).astype(int)
known_summary = known_summary.sort_values(["author_followers", "tweet_count"], ascending=False)
known_summary.to_csv(PROCESSED / "known_authors_activity_summary.csv", index=False)
known_summary.head(20)


In [ ]:
progress.step("Purpose")
fig = px.bar(
    known_summary.sort_values("author_followers", ascending=True).tail(25),
    x="author_followers", y="author_userName", orientation="h",
    hover_data=["tweet_count", "author_location", "author_isBlueVerified"],
    title="Top known public authors by follower count",
    labels={"author_followers": "Followers", "author_userName": "Author"}
)
fig.update_layout(height=760, yaxis_title="")
save_plotly(fig, INTERACTIVE / "03_known_authors_followers.html", FIGURES / "03_known_authors_followers.png")
fig.show()


In [ ]:
progress.step("Purpose")
fig = px.scatter(
    known_summary.query("tweet_count > 0"),
    x="author_followers", y="tweet_count",
    size="total_views", hover_name="author_userName",
    color="author_isBlueVerified",
    title="Known authors: followers vs tweet activity",
    labels={"author_followers": "Followers", "tweet_count": "Tweets in dataset"}
)
fig.update_layout(height=620, xaxis_type="log")
save_plotly(fig, INTERACTIVE / "03_known_authors_followers_vs_activity.html", FIGURES / "03_known_authors_followers_vs_activity.png")
fig.show()


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
